# 10 Model comparison on G0–G3

All six models side by side, read from `results/log.csv` (validation rows of the G0–G3 runs from notebooks 04 to
09). A cell is shown only when its model has been validated with all of seeds 0, 1 and 2 on that set; the first
table lists what is still missing. Nothing is refitted here.

The last section splits the validation block into returning and new customers (a `uid` already seen in the training
block or not), as the EDA recommended, using the validation scores saved in `results/preds/`.

In [ ]:
import json
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "src").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import average_precision_score

from src import experiment, feature_sets as fs

MODELS = ["logreg", "rf", "lgbm", "xgboost", "catboost", "mlp"]
LABELS = {"logreg": "Logistic regression", "rf": "Random forest", "lgbm": "LightGBM",
          "xgboost": "XGBoost", "catboost": "CatBoost", "mlp": "MLP"}
FEATURE_SETS = list(fs.FEATURE_SETS)
SEEDS = [0, 1, 2]
METRICS = {"pr_auc": "PR-AUC", "auroc": "AUROC", "recall_top1pct": "recall in top 1%"}
pd.set_option("display.width", 160)

## 1. What has been run

In [ ]:
log = pd.read_csv(experiment.LOG_PATH)
val = log[(log["split"] == "val") & log["model"].isin(MODELS) & log["feature_set"].isin(FEATURE_SETS)].copy()

def settings(p):
    d = json.loads(p)
    return json.dumps({k: v for k, v in d.items() if k not in ("threshold", "_fit")}, sort_keys=True)

val["settings"] = val["params"].map(settings)
status, rows = {}, []
for m in MODELS:
    for g in FEATURE_SETS:
        cell = val[(val["model"] == m) & (val["feature_set"] == g)]
        complete = [s for s, part in cell.groupby("settings") if sorted(part["seed"]) == SEEDS]
        if len(complete) > 1:
            raise ValueError(f"{m}/{g}: more than one set of settings validated with all seeds; "
                             "keep one in the log or choose explicitly")
        status[(m, g)] = "done" if complete else (f"seeds {sorted(cell['seed'])}" if len(cell) else "not run")
        if complete:
            rows.append(cell[cell["settings"] == complete[0]])
done = pd.concat(rows) if rows else val.iloc[:0]
pd.Series(status).unstack().reindex(index=MODELS, columns=FEATURE_SETS).rename(index=LABELS)

## 2. Validation results (mean ± sample sd over seeds 0, 1, 2)

In [ ]:
def table(metric):
    g = done.groupby(["model", "feature_set"])[metric]
    cell = g.mean().map("{:.4f}".format) + " ± " + g.std().map("{:.4f}".format)
    return cell.unstack().reindex(index=MODELS, columns=FEATURE_SETS).rename(index=LABELS)

for metric, label in METRICS.items():
    print(label)
    display(table(metric))

In [ ]:
mean = done.groupby(["model", "feature_set"])["pr_auc"].mean().unstack().reindex(index=MODELS, columns=FEATURE_SETS)
sd = done.groupby(["model", "feature_set"])["pr_auc"].std().unstack().reindex(index=MODELS, columns=FEATURE_SETS)
fig, ax = plt.subplots(figsize=(10, 4))
width = 0.8 / len(FEATURE_SETS)
x = np.arange(len(MODELS))
for i, g in enumerate(FEATURE_SETS):
    ax.bar(x + i * width, mean[g], width, yerr=sd[g].fillna(0), capsize=2, label=g)
ax.set_xticks(x + width * (len(FEATURE_SETS) - 1) / 2, [LABELS[m] for m in MODELS])
ax.set_ylabel("validation PR-AUC")
ax.set_title("Validation PR-AUC by model and feature set (mean ± sd over seeds)")
ax.legend(title="feature set")
ax.grid(axis="x")
fig.tight_layout()
plt.show()

**What each feature group adds:** the change in mean validation PR-AUC from one set to the next, per model.

In [ ]:
gain = mean.diff(axis=1).iloc[:, 1:]
gain.columns = [f"{a} → {b}" for a, b in zip(FEATURE_SETS[:-1], FEATURE_SETS[1:])]
gain["G0 → G3"] = mean["G3"] - mean["G0"]
gain.rename(index=LABELS).round(4)

## 3. Returning vs new customers (validation block)

A validation row is **returning** if its `uid` already appears in the training block, otherwise **new**. History
features (G2, G3) can only help returning customers, so the gap between the two groups shows where the customer
profile pays off. PR-AUC per group, averaged over the seeds whose scores are saved in `results/preds/`.

In [ ]:
spec = fs.load_spec()
ids = pd.read_parquet(ROOT / spec["parquet"], columns=["TransactionID", "uid"])
n_train = spec["train_block_rows"]
seen = set(ids["uid"].iloc[:n_train])
returning = ids.set_index("TransactionID")["uid"].isin(seen)

rows = []
for m in MODELS:
    for g in FEATURE_SETS:
        if status[(m, g)] != "done":
            continue
        scores = []
        for s in SEEDS:
            path = experiment.PREDS_DIR / f"{m}_{g}_val_seed{s}.parquet"
            if path.exists():
                p = pd.read_parquet(path)
                r = returning.reindex(p["TransactionID"]).to_numpy()
                scores.append({"returning": average_precision_score(p["isFraud"][r], p["score"][r]),
                               "new": average_precision_score(p["isFraud"][~r], p["score"][~r])})
        if scores:
            rows.append({"model": LABELS[m], "feature_set": g, "seeds": len(scores),
                         **pd.DataFrame(scores).mean().to_dict()})
by_customer = pd.DataFrame(rows)
if len(by_customer):
    val_rows = ids.iloc[n_train:n_train + len(ids) // 10]   # same block as src.split.holdout_indices
    print(f"validation rows from returning customers: {val_rows['uid'].isin(seen).mean():.1%}")
    display(by_customer.pivot(index="model", columns="feature_set", values=["returning", "new"]).round(4))
else:
    print("No saved validation scores yet: run notebooks 04 to 09 first.")

## 4. Findings

Write this once all cells are done, quoting only numbers from the tables above (they come from `results/log.csv`
and the saved validation scores):

- Which model and feature set score best on validation PR-AUC, and by how much over the next best.
- How much each feature group adds, and whether that is the same for every model.
- How returning and new customers differ.